In [1]:
import pennylane as qml
from pennylane import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# ---- Output paths ----
QUANTUM_ENCODER_PATH = "quantum_stock_encoder.pkl"
QUANTUM_FEATURES_PATH = "quantum_stock_features.pkl"

In [3]:
# Load dataset
data = pd.read_csv("C:\Infosys_smp\SHARE_MARKET_PREDICTION\dataset.csv")
# Standardize column names
data.columns = [c.strip().lower() for c in data.columns]
# Expected features
expected_features = ["open", "high", "low", "close", "volume"]
# Only take columns that exist
available_features = [f for f in expected_features if f in data.columns]
if len(available_features) == 0:
    raise ValueError("No expected numeric columns found in dataset.csv!")
print(f"Using columns: {available_features}")
# Extract values and fill missing data
X = data[available_features].fillna(0).values

Using columns: ['open', 'high', 'low', 'close', 'volume']


<>:2: SyntaxWarning: invalid escape sequence '\I'
<>:2: SyntaxWarning: invalid escape sequence '\I'
C:\Users\HP\AppData\Local\Temp\ipykernel_47708\3477804704.py:2: SyntaxWarning: invalid escape sequence '\I'
  data = pd.read_csv("C:\Infosys_smp\SHARE_MARKET_PREDICTION\dataset.csv")


In [4]:
# ---- Step 2: Normalize features (for angle embedding) ----
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
# Optional: normalize vectors to unit length
X_normalized = normalize(X_scaled, norm="l2")

In [5]:
# ---- Step 3: Quantum device setup ----
n_qubits = X_normalized.shape[1]  # number of features = number of qubits
dev = qml.device("default.qubit", wires=n_qubits)

In [6]:
# ---- Step 4: Quantum feature encoder ----
@qml.qnode(dev)
def quantum_encoder(x):
    """Encodes numeric market data into quantum states."""
    qml.templates.AngleEmbedding(x, wires=range(n_qubits))
    qml.templates.BasicEntanglerLayers(weights=np.ones((1, n_qubits)), wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [7]:
# ---- Step 5: Encode all rows ----
print("⚙️ Encoding stock data into quantum feature space...")
quantum_features = np.array([quantum_encoder(x) for x in X_normalized])

⚙️ Encoding stock data into quantum feature space...


In [8]:
# ---- Step 6: Save quantum components ----
joblib.dump(scaler, QUANTUM_ENCODER_PATH)
joblib.dump({
    "quantum_features": quantum_features,
    "X_normalized": X_normalized,
    "df": data
}, QUANTUM_FEATURES_PATH)

print(f"Quantum encoder saved to '{QUANTUM_ENCODER_PATH}'")
print(f"Quantum features saved to '{QUANTUM_FEATURES_PATH}'")

Quantum encoder saved to 'quantum_stock_encoder.pkl'
Quantum features saved to 'quantum_stock_features.pkl'


In [9]:
def compare_market_state(query_row_index, top_k=3):
    """
    Compares a given day's market state (query_row_index) with all others.
    Finds days with similar quantum-encoded patterns.
    """
    query_quantum = quantum_features[query_row_index]
    sims = cosine_similarity([query_quantum], quantum_features).flatten()
    top_idx = sims.argsort()[::-1][:top_k]

    print(f"\nQuantum Similarity for Market Day {query_row_index}:")
    print("Available columns:", data.columns.tolist())  # Debug helper

    for idx in top_idx:
        row = data.iloc[idx]
        date_val = row.get('Date', idx)
        open_val = row.get('Open', 'N/A')
        close_val = row.get('Close', 'N/A')
        volume_val = row.get('Volume', 'N/A')

        print(f"{date_val} | Similarity = {sims[idx]:.3f}")
        print(f"Open: {open_val} | Close: {close_val} | Volume: {volume_val}\n")


In [10]:
# Compare today’s market state (index 100) to other days
compare_market_state (query_row_index=100, top_k=3)


Quantum Similarity for Market Day 100:
Available columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'lag_1', 'lag_2', 'lag_3', 'lag_5', 'lag_7', 'roll_mean_3', 'roll_std_3', 'roll_mean_7', 'roll_std_7', 'roll_mean_14', 'roll_std_14', 'pct_change_1', 'vol_ma_7', 'target_close']
100 | Similarity = 1.000
Open: N/A | Close: N/A | Volume: N/A

71 | Similarity = 1.000
Open: N/A | Close: N/A | Volume: N/A

80 | Similarity = 1.000
Open: N/A | Close: N/A | Volume: N/A

